# Ablation Tahap 2 — Few-Shot k (Kaggle)

Notebook ini menjalankan **ablation few-shot k** untuk **GraphRAG dan Baseline**, lalu memilih k pemenang dengan **EX** (execution accuracy, dihitung in-process). Isinya hanya memanggil `src/experiments/ablation.py` per pipeline, jadi hasilnya identik dengan menjalankannya lewat CLI.

## Sebelum menjalankan
1. **Settings**: Accelerator **GPU T4**, **Internet On** (untuk `git clone` dan unduh model).
2. **Add Input**: dataset Spider (`alrette/spiderdataset`), supaya path default di `config.py` berlaku.
3. Isi **threshold** di sel 2 (hasil trial SLA). Sisanya biarkan.
4. Untuk run penuh gunakan **Save Version → Save & Run All (Commit)** supaya tetap jalan walau tab ditutup. Run LLM penuh bisa berjam-jam (batas sesi GPU Kaggle 12 jam).

## Alur
| Sel | Isi |
|---|---|
| 1 | Setup: clone repo, install dependency, cek GPU dan data |
| 2 | Konfigurasi (threshold, mode, k) |
| 3 | **Uji cepat** (5% dev set, satu k): gagal cepat kalau ada yang salah, hasilnya dibuang |
| 4 | **Run penuh** per pipeline (k = 0, 1, 3, 5) |
| 5 | Ringkasan EX per k dan **k pemenang** |
| 6 | Excel + zip untuk diunduh |

## Hasil
`/kaggle/working/hasil_ablation.xlsx` (sheet *Ablation Tahap 2*), `/kaggle/working/hasil_ablation.zip` (semua file mentah), dan di folder repo `outputs/tables/ablation_results.csv`, `ablation_ex_per_k_{graphrag,baseline}_skeleton.json`, `outputs/predictions/ablation_*`.

In [ ]:
# ============================================================
# 1. SETUP: clone repo, install dependency, cek GPU dan data
# Sel ini aman dijalankan ulang.
# ============================================================
import os, subprocess, sys, time
from pathlib import Path

REPO_URL = "https://github.com/william-darma-wijaya/thesis-2803.git"
REPO_BRANCH = "main"

def is_repo_root(p: Path) -> bool:
    return (p / "src" / "core" / "config.py").is_file()

def sh(cmd: str) -> None:
    print(f"$ {cmd}", flush=True)
    subprocess.run(cmd, shell=True, check=True)

if not is_repo_root(Path.cwd()):
    if is_repo_root(Path.cwd().parent):          # dijalankan dari notebooks/
        os.chdir("..")
    else:                                        # Kaggle: repo belum ada di /kaggle/working
        if not is_repo_root(Path("thesis-2803").resolve()):
            sh(f"git clone -b {REPO_BRANCH} {REPO_URL}")
        else:
            sh(f"git -C thesis-2803 fetch origin {REPO_BRANCH} && git -C thesis-2803 checkout {REPO_BRANCH} && git -C thesis-2803 pull origin {REPO_BRANCH}")
        os.chdir("thesis-2803")
REPO_ROOT = Path.cwd()
assert is_repo_root(REPO_ROOT), f"{REPO_ROOT} bukan root repo thesis-2803"
print("Repo root:", REPO_ROOT)
print("Branch   :", subprocess.run("git rev-parse --abbrev-ref HEAD", shell=True, capture_output=True, text=True).stdout.strip())
print("Commit   :", subprocess.run("git log --oneline -1", shell=True, capture_output=True, text=True).stdout.strip())

# torch, pandas, networkx, tqdm sudah ada di Kaggle -> jangan di-upgrade. nltk dibutuhkan parser SQL Spider.
sh("pip install -q -U 'bitsandbytes>=0.46.1' sentence-transformers transformers nltk openpyxl")
import nltk
for pkg in ("punkt", "punkt_tab"):
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f"[WARN] nltk.download({pkg}) gagal: {e}")

import torch
assert torch.cuda.is_available(), "GPU tidak aktif: Settings -> Accelerator -> GPU T4."
print("GPU      :", torch.cuda.get_device_name(0))

sys.path.insert(0, str(REPO_ROOT))
from src.core.config import PipelineConfig
_cfg = PipelineConfig()
assert _cfg.data_path.exists(), (
    f"Data Spider tidak ditemukan di {_cfg.data_path}. Tambahkan dataset Spider lewat 'Add Input' "
    "(alrette/spiderdataset), atau sesuaikan data_path di src/core/config.py.")
print("Data     :", _cfg.data_path)
print("Setup selesai.")

In [ ]:
# ============================================================
# 2. KONFIGURASI
# ============================================================
TABLE_THRESHOLD  = 0.50   # Stage 1: skor cosine minimum TABEL (dipakai GraphRAG DAN Baseline), hasil trial SLA
COLUMN_THRESHOLD = 0.70   # Stage 2: skor cosine minimum KOLOM (hanya GraphRAG), hasil trial SLA

MODES    = ["graphrag", "baseline"]   # pipeline yang di-ablasi; hapus salah satu kalau mau dipecah ke sesi lain
K_VALUES = None                       # None = few_shot_frame di config.py ([0, 1, 3, 5]); atau mis. [0, 1, 3]
RUN_SMOKE_TEST = True                 # uji cepat dulu (5% dev set, k=1): gagal cepat, hasilnya dibuang

cfg = PipelineConfig(table_similarity_threshold=TABLE_THRESHOLD, semantic_similarity_threshold=COLUMN_THRESHOLD)
K_LIST = K_VALUES or cfg.few_shot_frame
FLAGS = ["--table-threshold", str(TABLE_THRESHOLD), "--threshold", str(COLUMN_THRESHOLD)]

print(f"Threshold tabel / kolom : {TABLE_THRESHOLD} / {COLUMN_THRESHOLD}")
print(f"Seleksi few-shot        : {cfg.few_shot_selection}")
print(f"Pipeline                : {MODES}")
print(f"k yang diuji            : {K_LIST}")
print(f"LLM                     : {cfg.llm_model}")
print(f"max_ngram / stopword    : {cfg.max_ngram} / {cfg.query_stopword_mode}")

import re
def run(cmd):
    """Jalankan perintah, tampilkan outputnya langsung (progress bar tqdm diringkas), kembalikan lama (detik)."""
    print("\n$", " ".join(cmd), flush=True)
    t0, n_bar = time.time(), 0
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, "PYTHONUNBUFFERED": "1", "USE_TF": "0"})
    for line in p.stdout:
        if re.search(r"\d+%\|", line):          # baris progress tqdm: tampilkan tiap ke-100
            n_bar += 1
            if n_bar % 100:
                continue
        print(line, end="", flush=True)
    code = p.wait()
    dt = time.time() - t0
    if code != 0:
        raise RuntimeError(f"Gagal (exit {code}) setelah {dt/60:.1f} menit: {' '.join(cmd)}")
    print(f"[selesai dalam {dt/60:.1f} menit]")
    return dt

In [ ]:
# ============================================================
# 3. UJI CEPAT (5% dev set, k=1, GraphRAG): gagal cepat sebelum run berjam-jam
# ============================================================
import glob
if RUN_SMOKE_TEST:
    dt = run([sys.executable, "src/experiments/ablation.py", "--mode", "graphrag", "--sample", "0.05",
              "--k-values", "1", *FLAGS])
    print(f"\nUji cepat OK ({dt/60:.1f} menit, termasuk memuat model). Membuang hasil uji cepat ...")
    for f in glob.glob("outputs/predictions/ablation_*") + glob.glob("outputs/tables/ablation_*"):
        os.remove(f)
else:
    print("Uji cepat dilewati.")

In [ ]:
# ============================================================
# 4. RUN PENUH per pipeline (EX/ESM dihitung in-process untuk tiap k)
# Satu pipeline gagal tidak membatalkan yang lain; kegagalan dilaporkan di akhir sel.
# ============================================================
durations, failures = {}, {}
for mode in MODES:
    cmd = [sys.executable, "src/experiments/ablation.py", "--mode", mode, "--sample", "1.0",
           "--k-values", *map(str, K_LIST), *FLAGS]
    try:
        durations[mode] = run(cmd)
    except Exception as e:                                   # noqa: BLE001
        failures[mode] = str(e)
        print(f"\n[GAGAL] {mode}: {e}")

print("\n=== Ringkasan run ===")
for m in MODES:
    print(f"{m:<9}: " + (f"selesai {durations[m]/60:.1f} menit" if m in durations else f"GAGAL, {failures.get(m)}"))

In [ ]:
# ============================================================
# 5. RINGKASAN: EX per k dan k pemenang
# Aturan (src/experiments/ablation.py): k bersama = rata-rata EX kedua pipeline tertinggi; seri -> k lebih kecil.
# ============================================================
import pandas as pd
RESULTS_CSV = Path("outputs/tables/ablation_results.csv")
assert RESULTS_CSV.exists(), "ablation_results.csv belum ada: sel 4 gagal total, lihat pesan error di atas."
res = pd.read_csv(RESULTS_CSV)
if "few_shot_selection" not in res.columns:
    res["few_shot_selection"] = "question"
res = res[res["few_shot_selection"] == cfg.few_shot_selection]

ex = res.pivot_table(index="k", columns="mode", values="avg_ex", aggfunc="last").sort_index()
ex["rata_rata"] = ex.mean(axis=1)
display(ex.round(2).style.format("{:.2f}").highlight_max(subset=["rata_rata"], color="#d4edda"))

if {"graphrag", "baseline"} <= set(ex.columns[:-1]) and ex[["graphrag", "baseline"]].notna().all().all():
    best_k = int(ex["rata_rata"].round(6).sort_index().idxmax())   # idxmax mengambil kemunculan pertama = k terkecil
    print(f"\nk PEMENANG (rata-rata EX tertinggi, seri -> k lebih kecil): {best_k}")
    print(f"Tulis ke src/core/config.py: few_shot_k: int = {best_k}   (atau cfg.few_shot_k = {best_k} di eval_pipeline.ipynb)")
    print("Angka EX per k untuk Dimensi 6 (isi EX_PER_K di eval_pipeline.ipynb, GraphRAG):")
    print("   EX_PER_K = " + str({int(k): round(float(v), 2) for k, v in ex["graphrag"].items()}))
else:
    print("\nBelum ada hasil untuk kedua pipeline, jadi k bersama belum bisa ditentukan. Jalankan pipeline yang kurang.")

cols = ["mode", "k", "avg_ex", "avg_esm", "avg_prompt_tokens", "avg_output_tokens", "avg_token_consumption", "n_samples"]
display(res[[c for c in cols if c in res.columns]].sort_values(["mode", "k"]).reset_index(drop=True))

In [ ]:
# ============================================================
# 6. EXCEL + ZIP untuk diunduh dari panel Output Kaggle
# ============================================================
import shutil, zipfile
XLSX = Path("outputs/tables/hasil_ablation.xlsx")
run([sys.executable, "src/experiments/export_excel.py", "--skip-missing", "--out", str(XLSX)])

OUT_ROOT = Path("/kaggle/working")
if OUT_ROOT.exists():
    shutil.copy(XLSX, OUT_ROOT / "hasil_ablation.xlsx")
    with zipfile.ZipFile(OUT_ROOT / "hasil_ablation.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for d in ("outputs/tables", "outputs/predictions"):
            for f in Path(d).rglob("ablation_*"):
                z.write(f, f.as_posix())
        z.write(XLSX, XLSX.as_posix())
    print("Siap diunduh dari panel Output: hasil_ablation.xlsx, hasil_ablation.zip")